In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
# CARREGAR MODELOS OPENAI - Embeddings e CHAT

embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model_name='gpt-3.5-turbo', max_tokens=300)

In [11]:
pdf_link = './assets/anexo-projeto-de-lei.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [12]:
# SEPARAR EM CHUNKS

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4_000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [13]:
# SALVAR OS CHUNKS NO VECTOR DB

vectordb = Chroma(embedding_function=embeddings_model, persist_directory='naiveDB')

In [ ]:
# CARREGAR O DB

naive_retreiver = vectordb.as_retriever(search_kwargs={ 'k': 10 })

In [19]:
rerank = CohereRerank(model='rerank-multilingual-v3.0', top_n=3)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retreiver
)

In [23]:
TEMPLATE = """
Você é um especialista em legislação e tecnologia. Responda a pergunta abaixo utilizando o contexto informado.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [24]:
setup_retrival = RunnableParallel({
    'question': RunnablePassthrough(),
    'context': compressor_retriever,
})

output_parser = StrOutputParser()

compressor_retriever_chain = setup_retrival | rag_prompt | llm | output_parser

In [25]:
compressor_retriever_chain.invoke('Quais são os principais pontos de risco do marco legal de IA')

'Os principais pontos de risco do marco legal de IA estão relacionados à privacidade dos dados, viés algorítmico, transparência e responsabilidade. A falta de regras claras e consistentes pode resultar em uso inadequado de informações pessoais, discriminação injusta e dificuldade em responsabilizar os agentes responsáveis por danos causados pelo uso de sistemas de IA. É importante que a legislação estabeleça diretrizes sólidas para mitigar esses riscos e garantir a confiança dos usuários na tecnologia de IA.'